# Integração de bases públicas (Etapa 3)
Projeto de previsão de demanda de energia elétrica para apoio ao planejamento de capacidade da Copel.

Este notebook cruza o dataset tratado na Etapa 2 (carga diária do ONS no subsistema Sul) com quatro bases públicas.

| Fonte | O que traz | Granularidade | Chave de junção | Uso no projeto |
|---|---|---|---|---|
| ONS (Etapa 2) | Carga diária do Sul (MWmed) | diária | data | variável alvo |
| ANEEL, interrupções nas redes de distribuição | Cada interrupção da Copel com início, fim, consumidores afetados e causa | evento (segundos) | data de início | sinalizar dias com demanda reprimida (apagões) |
| ANEEL, indicadores DEC e FEC | Duração e frequência de interrupção por conjunto | mensal | ano e mês | validação cruzada e objetivo estratégico da Etapa 1 |
| EPE, consumo mensal por classe | Consumo (MWh) e consumidores por classe e UF | mensal | ano e mês | perfil residencial, industrial e comercial do Paraná |
| Open-Meteo (reanálise ERA5) | Temperatura diária das capitais do Sul | diária | data | variável explicativa climática |

As saídas ficam em `etapa3/saidas/`. São o dataset integrado, o dicionário com a linhagem de cada coluna, o relatório de qualidade por fonte, o registro de transformações, o registro de downloads com hash SHA-256 e um resumo com os números.

Para rodar no Google Colab, use Ambiente de execução > Executar tudo. Na primeira execução o notebook pede o arquivo `dataset_tratado_sul.csv` da Etapa 2. Os arquivos de interrupções da ANEEL são grandes, então o notebook processa um ano por vez e apaga o arquivo bruto depois de extrair os dados da Copel.

## 1. Configuração

In [ ]:
import hashlib, io, json, platform, re, shutil, urllib.request, zipfile
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.compute as pc
import pyarrow.parquet as pq
import matplotlib.pyplot as plt

# PARÂMETROS
ANOS_INTERRUPCOES = list(range(2017, 2027))   # anos da ANEEL
PADRAO_AGENTE = r"COPEL"                       # sigla buscada em SigAgente
UF_DISTRIBUIDORA = "PR"                        # UF da Copel na EPE
DEFASAGEM_MENSAL = 2                           # atraso das bases mensais (meses)
LIMIAR_FRACAO_AFETADOS = 0.20                  # grande interrupção a partir de 20% do PR
PERCENTIL_GRANDE_INTERRUPCAO = 0.99            # critério antigo, só para comparar
DURACAO_MAX_PLAUSIVEL_H = 72                   # acima disso só conta no relatório
MANTER_BRUTOS = False                          # apaga os brutos da ANEEL depois do filtro
REBAIXAR = False                               # True ignora o cache

CIDADES = {   # capitais do Sul
    "curitiba":      (-25.43, -49.27),
    "florianopolis": (-27.60, -48.55),
    "porto_alegre":  (-30.03, -51.23),
}

URL_INTERRUPCOES = {
    2017: "https://dadosabertos.aneel.gov.br/dataset/ccb25653-f07b-4f28-84c2-62a89d1f5a56/resource/92198821-d0a9-4279-985b-320e18c9e6c0/download/interrupcoes-energia-eletrica-2017.parquet",
    2018: "https://dadosabertos.aneel.gov.br/dataset/ccb25653-f07b-4f28-84c2-62a89d1f5a56/resource/d669e835-0b8e-47e4-94b4-a3012b3abfc6/download/interrupcoes-energia-eletrica-2018.parquet",
    2019: "https://dadosabertos.aneel.gov.br/dataset/ccb25653-f07b-4f28-84c2-62a89d1f5a56/resource/140e1704-9533-4004-b8ca-3d38cc8e4ba2/download/interrupcoes-energia-eletrica-2019.parquet",
    2020: "https://dadosabertos.aneel.gov.br/dataset/ccb25653-f07b-4f28-84c2-62a89d1f5a56/resource/19b932c7-08a4-48a2-a073-ca339b13deb3/download/interrupcoes-energia-eletrica-2020.parquet",
    2021: "https://dadosabertos.aneel.gov.br/dataset/ccb25653-f07b-4f28-84c2-62a89d1f5a56/resource/011e0086-8b2f-4fbc-a32b-f7c0f7bc9957/download/interrupcoes-energia-eletrica-2021.parquet",
    2022: "https://dadosabertos.aneel.gov.br/dataset/ccb25653-f07b-4f28-84c2-62a89d1f5a56/resource/f40b948c-81a3-4d56-8e35-0af9c2533178/download/interrupcoes-energia-eletrica-2022.parquet",
    2023: "https://dadosabertos.aneel.gov.br/dataset/ccb25653-f07b-4f28-84c2-62a89d1f5a56/resource/ddc26540-cd8c-4eef-a1ad-a234d24ed9c4/download/interrupcoes-energia-eletrica-2023.parquet",
    2024: "https://dadosabertos.aneel.gov.br/dataset/ccb25653-f07b-4f28-84c2-62a89d1f5a56/resource/fc5ca52c-329c-4443-a2d6-08ccec711ade/download/interrupcoes-energia-eletrica-2024.parquet",
    2025: "https://dadosabertos.aneel.gov.br/dataset/ccb25653-f07b-4f28-84c2-62a89d1f5a56/resource/691de320-cb3d-471b-b9ec-8c1b86af8c83/download/interrupcoes-energia-eletrica-2025.parquet",
    2026: "https://dadosabertos.aneel.gov.br/dataset/ccb25653-f07b-4f28-84c2-62a89d1f5a56/resource/cf722d0b-aa04-4681-bcd9-8a737e857182/download/interrupcoes-energia-eletrica-2026.parquet",
}
URL_DECFEC = "https://dadosabertos.aneel.gov.br/dataset/d5f0712e-62f6-4736-8dff-9991f10758a7/resource/4493985c-baea-429c-9df5-3030422c71d7/download/indicadores-continuidade-coletivos-2020-2029.zip"
URLS_EPE = [
    "https://www.epe.gov.br/sites-pt/publicacoes-dados-abertos/publicacoes/Documents/CONSUMO%20MENSAL%20DE%20ENERGIA%20EL%C3%89TRICA%20POR%20CLASSE.xlsx",
    "https://www.epe.gov.br/sites-pt/publicacoes-dados-abertos/publicacoes/PublicacoesArquivos/publicacao-190/CONSUMO%20MENSAL%20DE%20ENERGIA%20EL%C3%89TRICA%20POR%20CLASSE.xls",
]
URL_OPENMETEO = ("https://archive-api.open-meteo.com/v1/archive?latitude={lat}&longitude={lon}"
                 "&start_date={ini}&end_date={fim}"
                 "&daily=temperature_2m_mean,temperature_2m_max,temperature_2m_min"
                 "&timezone=America%2FSao_Paulo")

# PASTAS
BASE = Path("etapa3")
RAW = BASE / "dados_brutos"; OUT = BASE / "saidas"; FIG = BASE / "figuras"
for p in (RAW, OUT, FIG):
    p.mkdir(parents=True, exist_ok=True)
ARQ_ONS = Path("etapa2/saidas/dataset_tratado_sul.csv")

DATA_EXECUCAO = datetime.now()

plt.rcParams.update({
    "figure.figsize": (11, 4.2), "figure.dpi": 110, "savefig.dpi": 200,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#DDDDDD", "grid.linewidth": 0.6,
    "axes.titlesize": 12, "axes.titleweight": "bold", "font.size": 10,
})
COR_SERIE, COR_REF, COR_ALERTA, COR_2 = "#1F5F8B", "#7A7A7A", "#C0392B", "#D9822B"

print("Execução:", DATA_EXECUCAO.strftime("%Y-%m-%d %H:%M"), "(o Colab usa o horário UTC)")
print("Python", platform.python_version(), "| pandas", pd.__version__, "| numpy", np.__version__)

## 2. Funções de apoio

Download rastreável e registros de qualidade e de transformações. Toda fonte baixada fica registrada com URL, data e hash SHA-256, como na Etapa 2.

In [ ]:
registro_downloads, qualidade, log = [], [], []

def sha256(caminho):
    h = hashlib.sha256()
    with open(caminho, "rb") as f:
        for bloco in iter(lambda: f.read(1 << 20), b""):
            h.update(bloco)
    return h.hexdigest()

def arquivo_valido(caminho, sufixo=None):
    """Confere se o arquivo baixado está inteiro."""
    caminho = Path(caminho)
    if not caminho.exists() or caminho.stat().st_size == 0:
        return False
    suf = (sufixo or caminho.suffix).lower()
    try:
        if suf == ".parquet":
            with open(caminho, "rb") as f:
                ini = f.read(4); f.seek(-4, 2); fim = f.read(4)
            return ini == b"PAR1" and fim == b"PAR1"
        if suf in (".zip", ".xlsx"):
            return zipfile.is_zipfile(caminho)
        if suf == ".json":
            json.loads(caminho.read_text(encoding="utf-8"))
            return True
    except Exception:
        return False
    return True

def baixar(url, destino, fonte, tentativas=3):
    destino = Path(destino)
    status = "cache local"
    if destino.exists() and not arquivo_valido(destino):
        print(f"[{fonte}] {destino.name}: arquivo em cache incompleto ou corrompido, baixando de novo")
        destino.unlink()
    if REBAIXAR or not destino.exists():
        parcial = destino.with_name(destino.name + ".parcial")
        for t in range(1, tentativas + 1):
            try:
                req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0 (projeto academico UEL)"})
                with urllib.request.urlopen(req, timeout=600) as r, open(parcial, "wb") as f:
                    esperado = r.headers.get("Content-Length")
                    shutil.copyfileobj(r, f, length=1 << 20)
                recebido = parcial.stat().st_size
                if esperado and int(esperado) != recebido:
                    raise IOError(f"download incompleto: {recebido:,} de {int(esperado):,} bytes")
                if not arquivo_valido(parcial, destino.suffix):
                    raise IOError("arquivo recebido está corrompido")
                parcial.replace(destino)
                status = "baixado" if t == 1 else f"baixado na tentativa {t}"
                break
            except Exception as e:
                status = f"falhou: {e}"
                print(f"[{fonte}] {destino.name}: tentativa {t} de {tentativas} falhou ({e})")
                if parcial.exists():
                    parcial.unlink()
    info = {"fonte": fonte, "arquivo": destino.name, "url": url, "status": status,
            "data_obtencao": DATA_EXECUCAO.strftime("%Y-%m-%d %H:%M")}
    if destino.exists():
        info.update({"tamanho_bytes": destino.stat().st_size, "sha256": sha256(destino)})
    registro_downloads.append(info)
    print(f"[{fonte}] {destino.name}: {status}")
    return destino if destino.exists() else None

def registrar(fonte, dimensao, verificacao, ocorrencias, total, observacao=""):
    pct = round(100 * ocorrencias / total, 3) if total else None
    qualidade.append({"fonte": fonte, "dimensao": dimensao, "verificacao": verificacao,
                      "ocorrencias": ocorrencias, "total_avaliado": total, "percentual": pct,
                      "observacao": observacao})
    txt = f"{ocorrencias} de {total} ({pct:.2f}%)" if total else str(ocorrencias)
    print(f"[{fonte:<12}] [{dimensao:<12}] {verificacao}: {txt} {observacao}")

def transformar(fonte, dado_original, problema, transformacao, justificativa, afetados=""):
    log.append({"passo": len(log) + 1, "fonte": fonte, "dado_original": dado_original,
                "problema_identificado": problema, "transformacao": transformacao,
                "justificativa": justificativa, "registros_afetados": afetados})

def para_numero(s):
    if pd.api.types.is_numeric_dtype(s):
        return pd.to_numeric(s, errors="coerce")
    return pd.to_numeric(s.astype("string").str.strip().str.replace(",", ".", regex=False), errors="coerce")

FORMATOS_DATA_HORA = ["%d/%m/%Y %H:%M:%S", "%Y-%m-%d %H:%M:%S", "%Y-%m-%dT%H:%M:%S",
                      "%Y-%m-%d %H:%M:%S.%f", "%d/%m/%Y %H:%M", "%Y-%m-%d %H:%M", "%d/%m/%Y", "%Y-%m-%d"]

def para_data_hora(s):
    """Converte datas testando vários formatos."""
    if pd.api.types.is_datetime64_any_dtype(s):
        return s.dt.tz_localize(None) if getattr(s.dt, "tz", None) is not None else s
    s = s.astype("string").str.strip()
    out = pd.Series(pd.NaT, index=s.index, dtype="datetime64[ns]")
    for fmt in FORMATOS_DATA_HORA:
        falta = out.isna() & s.notna()
        if not falta.any():
            break
        conv = pd.to_datetime(s[falta], format=fmt, errors="coerce")
        out.loc[conv[conv.notna()].index] = conv[conv.notna()]
    return out

def primeira_coluna(colunas, candidatas):
    for c in candidatas:
        if c in colunas:
            return c
    return None

## 3. Base A, carga diária do ONS (dataset tratado da Etapa 2)

O notebook usa o arquivo gerado na Etapa 2, sem baixar o ONS de novo, para que a variável alvo seja a mesma documentada naquela etapa.

In [ ]:
if not ARQ_ONS.exists():
    ARQ_ONS.parent.mkdir(parents=True, exist_ok=True)
    try:
        from google.colab import files
        print("Envie o arquivo dataset_tratado_sul.csv gerado na Etapa 2:")
        enviado = files.upload()
        nome = next(iter(enviado))
        Path(nome).replace(ARQ_ONS)
    except ImportError:
        raise FileNotFoundError(f"Coloque o dataset da Etapa 2 em {ARQ_ONS}")

ons = pd.read_csv(ARQ_ONS, parse_dates=["data"])
registro_downloads.append({"fonte": "ONS (Etapa 2)", "arquivo": ARQ_ONS.name, "url": "dataset tratado da Etapa 2",
                           "status": "arquivo local", "data_obtencao": DATA_EXECUCAO.strftime("%Y-%m-%d %H:%M"),
                           "tamanho_bytes": ARQ_ONS.stat().st_size, "sha256": sha256(ARQ_ONS)})
ini_ons, fim_ons = ons["data"].min(), ons["data"].max()
registrar("ONS", "Completude", "Dias sem carga no dataset da Etapa 2", int(ons["carga_mwmed"].isna().sum()), len(ons))
registrar("ONS", "Unicidade", "Datas repetidas", int(ons["data"].duplicated().sum()), len(ons))
print(f"ONS: {len(ons):,} dias, {ini_ons:%d/%m/%Y} a {fim_ons:%d/%m/%Y}")

## 4. Base B, interrupções de energia da Copel (ANEEL)

Cada linha da base da ANEEL é uma interrupção em uma rede de distribuição do país. Para cada ano o notebook

1. lê só a coluna do agente, para descobrir as siglas da Copel
2. lê apenas as linhas e colunas da Copel, porque o arquivo inteiro não cabe com folga na memória
3. padroniza os nomes das colunas, porque o dicionário da ANEEL mudou em 2026 (por exemplo, `NumUnidadeConsumidora` virou `QtdConsumidoresAfetados`)

A medida principal é o CHI (consumidor-hora interrompido), que é o número de consumidores afetados vezes a duração em horas. Segue a lógica do DEC, sem dividir pelo total de consumidores.

In [ ]:
COLS = {
    "agente":    ["SigAgente"],
    "inicio":    ["DatInicioInterrupcao"],
    "fim":       ["DatFimInterrupcao"],
    "afetados":  ["QtdConsumidoresAfetados", "NumUnidadeConsumidora"],
    "ativos":    ["QtdConsumidoresAtivos", "NumConsumidorConjunto"],
    "tipo":      ["DscTipoInterrupcao", "DscFatoGeradorTipo"],
    "expurgo":   ["IdeMotivoExpurgo", "DscMotivoExpurgo"],
    "conjunto":  ["CodConjUnidadeConsumidora", "IdeConjuntoUnidadeConsumidora"],
    "causa":     ["DscFatoGeradorCausa", "FatGeradorInterrupcao"],
}

def extrair_copel(caminho, ano):
    nomes = pq.ParquetFile(caminho).schema_arrow.names
    mapa = {k: primeira_coluna(nomes, v) for k, v in COLS.items()}
    faltam = [k for k in ("agente", "inicio", "fim", "afetados") if mapa[k] is None]
    if faltam:
        raise KeyError(f"{ano}: colunas obrigatórias não encontradas {faltam}. Colunas do arquivo: {nomes}")
    col_agente = pq.read_table(caminho, columns=[mapa["agente"]]).column(0)
    total_linhas = len(col_agente)
    unicos = pc.unique(col_agente).to_pylist()
    siglas = sorted(str(a) for a in unicos if a is not None and re.search(PADRAO_AGENTE, str(a), re.IGNORECASE))
    del col_agente
    usar = [c for c in mapa.values() if c]
    tab = pq.read_table(caminho, columns=usar, filters=[(mapa["agente"], "in", list(siglas))]).to_pandas()
    tab = tab.rename(columns={v: k for k, v in mapa.items() if v})
    for k in COLS:
        if k not in tab.columns:
            tab[k] = pd.NA
    tab["ano_arquivo"] = ano
    esquema = "2026+ (novo)" if "QtdConsumidoresAfetados" in nomes else "até 2025 (antigo)"
    return tab, total_linhas, siglas, esquema

extratos, info_anos = [], []
for ano in ANOS_INTERRUPCOES:
    cache = RAW / f"interrupcoes_copel_{ano}.csv.gz"
    if cache.exists() and not REBAIXAR:
        tab = pd.read_csv(cache, dtype=str)
        info_anos.append({"ano": ano, "origem": "extrato em cache", "linhas_brasil": None,
                          "linhas_copel": len(tab), "siglas": "", "esquema": ""})
        print(f"[ANEEL] {ano}: extrato da Copel em cache ({len(tab):,} linhas)")
        extratos.append(tab); continue
    arq = baixar(URL_INTERRUPCOES[ano], RAW / f"interrupcoes-energia-eletrica-{ano}.parquet", "ANEEL interrupções")
    if arq is None:
        info_anos.append({"ano": ano, "origem": "falhou", "linhas_brasil": None, "linhas_copel": 0, "siglas": "", "esquema": ""})
        registrar("ANEEL interr.", "Completude", f"Arquivo de {ano} não baixado (ano ignorado nesta execução)", 1, 1,
                  "rode a célula de novo para tentar outra vez")
        continue
    try:
        tab, total, siglas, esquema = extrair_copel(arq, ano)
    except Exception as e:
        print(f"[ANEEL] {ano}: não foi possível ler o arquivo ({e}). O ano fica de fora; rode a célula de novo para tentar outra vez.")
        registrar("ANEEL interr.", "Completude", f"Arquivo de {ano} ilegível (ano ignorado nesta execução)", 1, 1, str(e)[:150])
        info_anos.append({"ano": ano, "origem": "falhou na leitura", "linhas_brasil": None, "linhas_copel": 0, "siglas": "", "esquema": ""})
        if arq.exists():
            arq.unlink()
        continue
    tab.astype("string").to_csv(cache, index=False, compression="gzip")
    info_anos.append({"ano": ano, "origem": "parquet ANEEL", "linhas_brasil": total, "linhas_copel": len(tab),
                      "siglas": ", ".join(siglas), "esquema": esquema})
    print(f"[ANEEL] {ano}: {total:,} interrupções no Brasil, {len(tab):,} da Copel ({', '.join(siglas)}), esquema {esquema}")
    extratos.append(tab.astype("string"))
    if not MANTER_BRUTOS:
        arq.unlink()

df_anos = pd.DataFrame(info_anos)
interrup = pd.concat(extratos, ignore_index=True) if extratos else pd.DataFrame(columns=list(COLS) + ["ano_arquivo"])
transformar("ANEEL interrupções", f"{len(extratos)} arquivos anuais (Brasil inteiro)", "Base nacional, com colunas renomeadas em 2026",
            f"Filtro do agente por '{PADRAO_AGENTE}' e padronização das colunas dos dois dicionários",
            "Escopo da organização (Copel) e série contínua apesar da mudança de esquema", len(interrup))
df_anos

### 4.1 Qualidade e cálculo do CHI

In [ ]:
iv = interrup.copy()
iv["inicio"] = para_data_hora(iv["inicio"])
iv["fim"] = para_data_hora(iv["fim"])
iv["afetados"] = para_numero(iv["afetados"])
iv["ativos"] = para_numero(iv["ativos"])
N_I = len(iv)

registrar("ANEEL interr.", "Validade", "Início não reconhecível como data", int(iv["inicio"].isna().sum()), N_I)
registrar("ANEEL interr.", "Validade", "Fim não reconhecível como data", int(iv["fim"].isna().sum()), N_I)
iv["duracao_h"] = (iv["fim"] - iv["inicio"]).dt.total_seconds() / 3600
neg = iv["duracao_h"] < 0
registrar("ANEEL interr.", "Consistência", "Fim anterior ao início", int(neg.sum()), N_I)
longas = iv["duracao_h"] > DURACAO_MAX_PLAUSIVEL_H
registrar("ANEEL interr.", "Acurácia", f"Duração maior que {DURACAO_MAX_PLAUSIVEL_H} h (mantidas)", int(longas.sum()), N_I)
registrar("ANEEL interr.", "Completude", "Consumidores afetados ausentes", int(iv["afetados"].isna().sum()), N_I)
registrar("ANEEL interr.", "Validade", "Consumidores afetados menor ou igual a zero", int((iv["afetados"] <= 0).sum()), N_I)
fora = iv["inicio"].notna() & (iv["inicio"].dt.year.astype("Int64") != pd.to_numeric(iv["ano_arquivo"], errors="coerce").astype("Int64"))
registrar("ANEEL interr.", "Consistência", "Início fora do ano do arquivo", int(fora.sum()), N_I)
chave_dup = ["conjunto", "inicio", "fim", "afetados"]
dup = iv.duplicated(subset=chave_dup, keep="first") & iv["inicio"].notna()
registrar("ANEEL interr.", "Unicidade", "Interrupções repetidas (conjunto, início, fim, afetados)", int(dup.sum()), N_I)

iv = iv[~dup & iv["inicio"].notna()].copy()
iv.loc[iv["duracao_h"] < 0, "duracao_h"] = np.nan
iv["chi"] = iv["afetados"] * iv["duracao_h"]
tipo = iv["tipo"].astype("string").str.upper()
iv["nao_programada"] = tipo.str.contains(r"N[ÃA]O", regex=True).fillna(True)
exp = iv["expurgo"].astype("string").str.upper()
iv["dia_critico"] = exp.str.fullmatch(r"\s*6(\.0)?\s*").fillna(False) | exp.str.contains(r"CR[IÍ]TICO", regex=True).fillna(False)
novo = pd.to_numeric(iv["ano_arquivo"], errors="coerce") >= 2026
for nome_esq, m in (("até 2025 (código numérico)", ~novo), ("2026 (texto)", novo)):
    dias = iv.loc[m].groupby(iv.loc[m, "inicio"].dt.normalize())["dia_critico"].max()
    registrar("ANEEL interr.", "Consistência", f"Dias com expurgo por dia crítico, esquema {nome_esq}",
              int(dias.sum()), len(dias), "dia crítico é definido por conjunto, não para a distribuidora inteira")
transformar("ANEEL interrupções", "Motivo de expurgo (dia crítico)", "Campo não comparável entre os esquemas, código até 2025 e texto em 2026",
            "Indicador de dia crítico descartado", "Uma variável com significado diferente em cada período induziria o modelo a erro", int(iv["dia_critico"].sum()))
transformar("ANEEL interrupções", "Interrupções com início e fim", "Medida comparável de impacto",
            "CHI = consumidores afetados x duração (h), duração negativa vira nula e repetidas são removidas",
            "Resume em um número o tamanho do apagão, na mesma lógica do DEC", int(dup.sum()))

iv["data"] = iv["inicio"].dt.normalize()
diario_int = iv.groupby("data").agg(
    n_interrupcoes=("chi", "size"),
    n_interrupcoes_nao_prog=("nao_programada", "sum"),
    consumidores_afetados=("afetados", "sum"),
    chi_total=("chi", "sum"),
    chi_nao_prog=("chi", lambda x: x[iv.loc[x.index, "nao_programada"]].sum()),
)
cobertura_int = (diario_int.index.min(), diario_int.index.max()) if len(diario_int) else (pd.NaT, pd.NaT)

# mês quase vazio é lacuna da fonte
meses_lacuna = []
if len(diario_int):
    por_mes = diario_int["n_interrupcoes"].groupby(diario_int.index.to_period("M")).sum()
    todos = pd.period_range(cobertura_int[0].to_period("M"), cobertura_int[1].to_period("M"), freq="M")
    por_mes = por_mes.reindex(todos, fill_value=0)
    limite_mes = 0.05 * por_mes.median()
    meses_lacuna = [m for m, v in por_mes.items() if v < limite_mes]
    registrar("ANEEL interr.", "Completude", "Meses sem registros da Copel dentro do período coberto",
              len(meses_lacuna), len(todos), ", ".join(m.strftime("%m/%Y") for m in meses_lacuna) or "nenhum")
    if meses_lacuna:
        transformar("ANEEL interrupções", "Meses sem nenhuma interrupção registrada", "Lacuna da fonte (a Copel tem centenas de interrupções por dia)",
                    "Dias desses meses marcados como sem dados (tem_dados_interrupcao = 0), e não como zero interrupções",
                    "Preencher com zero diria que não houve interrupções, o que é falso", len(meses_lacuna))

# comparação entre os dois esquemas
mil = lambda x: f"{x:,.0f}".replace(",", ".")
dec2 = lambda x: f"{x:.2f}".replace(".", ",")
if len(diario_int):
    esq_novo = diario_int.index.year >= 2026
    if esq_novo.any() and (~esq_novo).any():
        a_ant, a_nov = diario_int.loc[~esq_novo, "consumidores_afetados"].median(), diario_int.loc[esq_novo, "consumidores_afetados"].median()
        dur = lambda m: diario_int.loc[m, "chi_total"].sum() / diario_int.loc[m, "consumidores_afetados"].sum()
        qualidade.append({"fonte": "ANEEL interr.", "dimensao": "Consistência",
                          "verificacao": "Comparabilidade entre esquemas (mediana diária de consumidores afetados e duração média)",
                          "ocorrencias": None, "total_avaliado": None, "percentual": round(100 * a_nov / a_ant, 1),
                          "observacao": f"afetados por dia {mil(a_ant)} até 2025 contra {mil(a_nov)} em 2026 ({100 * a_nov / a_ant:.0f}%), "
                                        f"duração média de {dec2(dur(~esq_novo))} h contra {dec2(dur(esq_novo))} h. "
                                        "Diferença grande indica mudança na forma de contar e não no serviço"})
        print(f"Afetados por dia (mediana): {a_ant:,.0f} até 2025 x {a_nov:,.0f} em 2026; duração média {dur(~esq_novo):.2f} h x {dur(esq_novo):.2f} h")
transformar("ANEEL interrupções", "Eventos com data e hora", "Granularidade diferente da carga (diária)",
            "Agregação por data de início com contagem, consumidores afetados, CHI total e não programado",
            "Permite a junção com a carga diária. Interrupções que atravessam a meia-noite ficam no dia em que começaram", len(diario_int))
print(f"Interrupções da Copel: {len(iv):,} eventos, {len(diario_int):,} dias, "
      f"{cobertura_int[0]:%d/%m/%Y} a {cobertura_int[1]:%d/%m/%Y}")
diario_int.describe().T[["mean", "50%", "max"]].round(1)

## 5. Base C, indicadores DEC e FEC da Copel (ANEEL)

O DEC é a duração média de interrupção por consumidor (horas) e o FEC é a frequência média (vezes), por conjunto elétrico. A base não traz o número de consumidores de cada conjunto, então usamos a média simples entre conjuntos. Ela serve para validar o CHI calculado na seção 4 e para acompanhar o indicador citado como objetivo estratégico na Etapa 1.

In [ ]:
arq = baixar(URL_DECFEC, RAW / "indicadores-continuidade-coletivos-2020-2029.zip", "ANEEL DEC/FEC")

def ler_csv_flexivel(binario):
    for enc in ("utf-8-sig", "latin-1"):
        for sep in (";", ","):
            try:
                df = pd.read_csv(io.BytesIO(binario), sep=sep, encoding=enc, dtype=str)
            except (UnicodeDecodeError, pd.errors.ParserError):
                continue
            if df.shape[1] >= 5:
                return df
    raise ValueError("Não foi possível ler o CSV")

decfec_mensal = pd.DataFrame()
if arq is not None:
    with zipfile.ZipFile(arq) as z:
        nomes_csv = [n for n in z.namelist() if n.lower().endswith(".csv")]
        dec = pd.concat([ler_csv_flexivel(z.read(n)) for n in nomes_csv], ignore_index=True)
    print(f"DEC/FEC: {len(nomes_csv)} arquivo(s) CSV no zip, {len(dec):,} linhas")
    dec.columns = [c.strip() for c in dec.columns]
    copel = dec[dec["SigAgente"].astype("string").str.contains(PADRAO_AGENTE, case=False, regex=True)].copy()
    copel["SigIndicador"] = copel["SigIndicador"].astype("string").str.strip().str.upper()
    copel["mes"] = para_numero(copel["NumPeriodoIndice"])
    copel["ano"] = para_numero(copel["AnoIndice"])
    copel["valor"] = para_numero(copel["VlrIndiceEnviado"])
    ind = copel["SigIndicador"].value_counts()
    per = sorted(copel["mes"].dropna().unique().astype(int))
    print("Indicadores da Copel na base:", dict(ind.head(10)))
    print("Valores de NumPeriodoIndice:", per[:20])
    qualidade.append({"fonte": "ANEEL DEC/FEC", "dimensao": "Consistência", "verificacao": "Indicadores e períodos presentes (Copel)",
                      "ocorrencias": None, "total_avaliado": len(copel), "percentual": None,
                      "observacao": "indicadores " + ", ".join(f"{k}={v}" for k, v in ind.head(8).items())
                                    + " e períodos " + ", ".join(map(str, per[:16]))})
    registrar("ANEEL DEC/FEC", "Completude", "Valores de indicador ausentes (Copel)", int(copel["valor"].isna().sum()), len(copel))
    registrar("ANEEL DEC/FEC", "Validade", "Valores negativos (Copel)", int((copel["valor"] < 0).sum()), len(copel))
    mensal = copel[copel["SigIndicador"].isin(["DEC", "FEC"]) & copel["mes"].between(1, 12)]
    decfec_mensal = (mensal.groupby(["ano", "mes", "SigIndicador"])["valor"].mean().unstack("SigIndicador")
                     .rename(columns={"DEC": "dec_medio_conjuntos", "FEC": "fec_medio_conjuntos"}))
    decfec_mensal["n_conjuntos"] = mensal.groupby(["ano", "mes"])["IdeConjUndConsumidoras"].nunique()
    decfec_mensal.index = pd.to_datetime(dict(year=decfec_mensal.index.get_level_values(0).astype(int),
                                              month=decfec_mensal.index.get_level_values(1).astype(int), day=1))
    decfec_mensal.index.name = "mes_ref"
    transformar("ANEEL DEC/FEC", "DEC e FEC por conjunto e mês", "Granularidade por conjunto, sem peso de consumidores",
                "Filtro da Copel, indicadores DEC e FEC mensais (períodos 1 a 12), média simples entre conjuntos",
                "Indicador mensal da distribuidora para validação e acompanhamento", len(mensal))
    print(f"DEC/FEC mensal da Copel: {len(decfec_mensal)} meses")
decfec_mensal.tail()

## 6. Base D, consumo mensal por classe (EPE)

A planilha da EPE traz uma aba por classe (por exemplo `CONSUMO RESIDENCIAL POR UF`), com os estados nas linhas e os meses nas colunas. Esse formato é diferente do descrito no dicionário de dados da própria EPE, e isso fica anotado no relatório de qualidade.

O leitor abaixo acha em cada aba a linha do estado e o cabeçalho de meses (datas, ou ano e mês) sem depender da posição exata das células. Usamos o Paraná (área da Copel) por classe e a soma de PR, SC e RS para comparar com a carga do ONS. Na planilha da EPE a classe "outros" já inclui o rural.

In [ ]:
arq_epe = None
for url in URLS_EPE:
    ext = ".xlsx" if url.endswith(".xlsx") else ".xls"
    arq_epe = baixar(url, RAW / f"epe_consumo_mensal_por_classe{ext}", "EPE")
    if arq_epe is not None:
        break

MESES_PT = {"JAN": 1, "FEV": 2, "MAR": 3, "ABR": 4, "MAI": 5, "JUN": 6,
            "JUL": 7, "AGO": 8, "SET": 9, "OUT": 10, "NOV": 11, "DEZ": 12}
PADRAO_UF = {"PR": r"^\s*(PR|PARAN[ÁA])\b", "SC": r"^\s*(SC|SANTA\s+CATARINA)\b", "RS": r"^\s*(RS|RIO\s+GRANDE\s+DO\s+SUL)\b"}

def _mes_de(v):
    """Lê um cabeçalho como mês."""
    if isinstance(v, (pd.Timestamp, datetime)):
        return pd.Timestamp(v).to_period("M").to_timestamp()
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return None
    t = str(v).strip().upper()
    m = re.match(r"^(JAN|FEV|MAR|ABR|MAI|JUN|JUL|AGO|SET|OUT|NOV|DEZ)[A-Z]*[\s/\-\.]*(\d{2,4})$", t)
    if m:
        ano = int(m.group(2)); ano = ano + 2000 if ano < 100 else ano
        return pd.Timestamp(year=ano, month=MESES_PT[m.group(1)], day=1)
    m = re.match(r"^(\d{4})[\-/](\d{1,2})", t)
    if m and 1 <= int(m.group(2)) <= 12:
        return pd.Timestamp(year=int(m.group(1)), month=int(m.group(2)), day=1)
    return None

def _so_mes(v):
    t = str(v).strip().upper()[:3] if v is not None else ""
    return MESES_PT.get(t)

def _ano(v):
    try:
        a = int(float(str(v).strip()))
        return a if 1990 <= a <= 2100 else None
    except ValueError:
        return None

def _cabecalho_colunas(bruto, linha_alvo, col_rotulo):
    """Acha o cabeçalho de meses acima da linha do estado."""
    for h in range(linha_alvo - 1, -1, -1):
        linha = bruto.iloc[h]
        datas = {j: _mes_de(linha.iloc[j]) for j in range(col_rotulo + 1, bruto.shape[1])}
        datas = {j: d for j, d in datas.items() if d is not None}
        if len(datas) >= 6:
            return datas
        meses = {j: _so_mes(linha.iloc[j]) for j in range(col_rotulo + 1, bruto.shape[1])}
        meses = {j: m for j, m in meses.items() if m}
        if len(meses) >= 6:
            # ano em linha acima, com célula mesclada
            for a in range(h - 1, max(-1, h - 4), -1):
                anos = [_ano(x) for x in bruto.iloc[a].tolist()]
                if sum(x is not None for x in anos) >= 1:
                    anos_ff, ultimo = [], None
                    for x in anos:
                        ultimo = x if x is not None else ultimo
                        anos_ff.append(ultimo)
                    out = {j: pd.Timestamp(year=int(anos_ff[j]), month=m, day=1)
                           for j, m in meses.items() if anos_ff[j] is not None}
                    if out:
                        return out
            # ano como título do bloco
            for a in range(h - 1, max(-1, h - 6), -1):
                ano = next((_ano(x) for x in bruto.iloc[a].tolist() if _ano(x)), None)
                if ano:
                    return {j: pd.Timestamp(year=ano, month=m, day=1) for j, m in meses.items()}
    return {}

def serie_uf(xls, aba, uf):
    """Série mensal de um estado em uma aba da EPE."""
    bruto = pd.read_excel(xls, sheet_name=aba, header=None)
    padrao = re.compile(PADRAO_UF[uf], re.IGNORECASE)
    pontos = {}
    for c in range(min(4, bruto.shape[1])):
        for r in bruto.index[bruto.iloc[:, c].astype("string").str.match(padrao, na=False)]:
            cab = _cabecalho_colunas(bruto, r, c)
            for j, mes in cab.items():
                v = para_numero(pd.Series([bruto.iat[r, j]])).iloc[0]
                if pd.notna(v):
                    pontos[mes] = v
        if pontos:
            break
    if not pontos:   # layout transposto
        for h in range(min(15, len(bruto))):
            cols = [j for j in range(bruto.shape[1]) if padrao.match(str(bruto.iat[h, j]))]
            if cols:
                j = cols[0]
                for r in range(h + 1, len(bruto)):
                    mes = next((_mes_de(bruto.iat[r, k]) for k in range(min(3, bruto.shape[1])) if _mes_de(bruto.iat[r, k])), None)
                    v = para_numero(pd.Series([bruto.iat[r, j]])).iloc[0]
                    if mes is not None and pd.notna(v):
                        pontos[mes] = v
                break
    return pd.Series(pontos, dtype=float).sort_index()

def achar_aba(xls, *termos):
    for s in xls.sheet_names:
        u = s.upper()
        if all(t in u for t in termos):
            return s
    return None

epe_pr, epe_sul = pd.DataFrame(), pd.DataFrame()
if arq_epe is not None:
    try:
        xls = pd.ExcelFile(arq_epe)
        print("Abas:", xls.sheet_names)
        abas = {
            "total": achar_aba(xls, "CONSUMO POR UF"),
            "residencial": achar_aba(xls, "CONSUMO RESIDENCIAL", "UF"),
            "industrial": achar_aba(xls, "CONSUMO INDUSTRIAL", "UF"),
            "comercial": achar_aba(xls, "CONSUMO COMERCIAL", "UF"),
            "outros": achar_aba(xls, "CONSUMO OUTROS", "UF"),
        }
        abas_cons = [achar_aba(xls, "CONSUMIDORES", t, "UF") for t in ("RES", "INDUSTRIA", "COMERCIA", "OUTROS")]
        faltando = [k for k, v in abas.items() if v is None]
        registrar("EPE", "Consistência", "Abas por UF esperadas e não encontradas", len(faltando), len(abas), ", ".join(faltando))
        qualidade.append({"fonte": "EPE", "dimensao": "Consistência", "verificacao": "Formato da planilha x dicionário de dados da EPE",
                          "ocorrencias": None, "total_avaliado": None, "percentual": None,
                          "observacao": "planilha em formato largo (uma aba por classe, meses em colunas), diferente do formato longo descrito no dicionário"})
        series = {}
        for classe, aba in abas.items():
            if aba:
                s = serie_uf(xls, aba, UF_DISTRIBUIDORA)
                series[f"consumo_pr_{classe}_mwh"] = s
                print(f"  {aba}: {len(s)} meses do {UF_DISTRIBUIDORA}"
                      + (f" ({s.index.min():%m/%Y} a {s.index.max():%m/%Y})" if len(s) else ""))
        epe_pr = pd.DataFrame(series)
        epe_pr.index.name = "mes_ref"
        cons = [serie_uf(xls, a, UF_DISTRIBUIDORA) for a in abas_cons if a]
        if cons:
            epe_pr["consumidores_pr"] = pd.concat(cons, axis=1).sum(axis=1, min_count=len(cons))
        if "consumo_pr_total_mwh" not in epe_pr:
            partes = [c for c in epe_pr if c.startswith("consumo_pr_")]
            epe_pr["consumo_pr_total_mwh"] = epe_pr[partes].sum(axis=1, min_count=len(partes))
        # valores pequenos indicam GWh
        cols_cons = [c for c in epe_pr if c.startswith("consumo_pr_")]
        mediana = epe_pr["consumo_pr_total_mwh"].median()
        fator = 1000 if mediana < 1e5 else 1
        if fator != 1:
            epe_pr[cols_cons] = epe_pr[cols_cons] * fator
            transformar("EPE", "Consumo em GWh", "Unidade diferente das demais bases (MWh)", "Multiplicação por 1000",
                        "Padroniza a unidade em MWh", len(epe_pr))
        for c in ("residencial", "industrial", "comercial", "outros"):
            col = f"consumo_pr_{c}_mwh"
            if col in epe_pr:
                epe_pr[f"part_{c}_pr"] = epe_pr[col] / epe_pr["consumo_pr_total_mwh"]
        soma_classes = epe_pr[[c for c in cols_cons if c != "consumo_pr_total_mwh"]].sum(axis=1, min_count=1)
        dif = (soma_classes / epe_pr["consumo_pr_total_mwh"] - 1).abs()
        registrar("EPE", "Consistência", "Meses em que a soma das classes difere do total do PR em mais de 1%",
                  int((dif > 0.01).sum()), int(dif.notna().sum()))
        registrar("EPE", "Completude", f"Meses do {UF_DISTRIBUIDORA} com alguma classe ausente",
                  int(epe_pr[cols_cons].isna().any(axis=1).sum()), len(epe_pr))
        if abas["total"]:
            sul = pd.concat([serie_uf(xls, abas["total"], u) for u in ("PR", "SC", "RS")], axis=1)
            epe_sul = (sul.sum(axis=1, min_count=3) * fator).to_frame("consumo_sul_epe_mwh")
        transformar("EPE", "Abas por classe com estados nas linhas e meses nas colunas", "Formato largo, uma tabela por classe",
                    f"Extração da linha do {UF_DISTRIBUIDORA} em cada aba, uma coluna por classe e participações. Sul = PR + SC + RS",
                    "Perfil de consumo da área da Copel e total do Sul para validação", len(epe_pr))
        print(f"EPE: {UF_DISTRIBUIDORA} {epe_pr.index.min():%m/%Y} a {epe_pr.index.max():%m/%Y}, "
              f"mediana do consumo total {epe_pr['consumo_pr_total_mwh'].median():,.0f} MWh/mês")
    except Exception as e:
        epe_pr, epe_sul = pd.DataFrame(), pd.DataFrame()
        registrar("EPE", "Consistência", "Falha ao ler a planilha da EPE (base ignorada nesta execução)", 1, 1, str(e)[:200])
        print("ATENÇÃO: a planilha da EPE não pôde ser lida e foi ignorada. O restante do notebook continua.")
        print("Erro:", e)
epe_pr.tail(3).round(3)

## 7. Base E, temperatura diária (Open-Meteo)

Temperaturas média, máxima e mínima diárias das três capitais do Sul, da reanálise ERA5 (licença CC-BY 4.0), pela API gratuita da Open-Meteo. Substitui o download manual do INMET previsto na Etapa 2 e não tem falhas de estação.

In [ ]:
temps = {}
for cid, (lat, lon) in CIDADES.items():
    url = URL_OPENMETEO.format(lat=lat, lon=lon, ini=f"{ini_ons:%Y-%m-%d}", fim=f"{fim_ons:%Y-%m-%d}")
    arq = baixar(url, RAW / f"openmeteo_{cid}.json", "Open-Meteo")
    if arq is None:
        continue
    d = json.loads(Path(arq).read_text())["daily"]
    t = pd.DataFrame(d).rename(columns={"time": "data"})
    t["data"] = pd.to_datetime(t["data"])
    temps[cid] = t.set_index("data")

temperatura = pd.DataFrame()
if temps:
    for cid, t in temps.items():
        registrar("Open-Meteo", "Completude", f"Dias sem temperatura média ({cid})",
                  int(t["temperature_2m_mean"].isna().sum()), len(t))
    temperatura["temp_media_sul"] = pd.concat([t["temperature_2m_mean"] for t in temps.values()], axis=1).mean(axis=1)
    temperatura["temp_max_sul"] = pd.concat([t["temperature_2m_max"] for t in temps.values()], axis=1).mean(axis=1)
    temperatura["temp_min_sul"] = pd.concat([t["temperature_2m_min"] for t in temps.values()], axis=1).mean(axis=1)
    if "curitiba" in temps:
        temperatura["temp_media_curitiba"] = temps["curitiba"]["temperature_2m_mean"]
    temperatura["graus_dia_calor"] = (temperatura["temp_media_sul"] - 24).clip(lower=0)
    temperatura["graus_dia_frio"] = (18 - temperatura["temp_media_sul"]).clip(lower=0)
    transformar("Open-Meteo", "Temperatura diária de 3 capitais", "Três séries para uma região",
                "Média das 3 capitais e graus-dia de calor (acima de 24 °C) e de frio (abaixo de 18 °C)",
                "Captura o efeito do ar-condicionado no verão e do aquecimento no inverno", len(temperatura))
temperatura.describe().T.round(1)

## 8. Integração

Regras de governança usadas na junção

- A base diária é o calendário do ONS da Etapa 2. As junções são à esquerda, então nenhum dia de carga se perde.
- As interrupções só existem a partir de 2017. Antes disso as colunas ficam vazias e `tem_dados_interrupcao` indica a cobertura. Dias de 2017 em diante sem nenhuma interrupção recebem zero.
- As bases mensais (EPE e DEC/FEC) são publicadas semanas depois do fim do mês. Para não usar uma informação que ainda não existiria no momento da previsão, entram com defasagem de 2 meses (colunas `_lag2m`). As versões sem defasagem ficam só para análise.

In [ ]:
integ = ons.set_index("data").copy()

# interrupções
integ = integ.join(diario_int, how="left")
inicio_int = cobertura_int[0]
anos_ok = set(df_anos.loc[pd.to_numeric(df_anos["linhas_copel"], errors="coerce").fillna(0) > 0, "ano"].astype(int))
anos_faltando = sorted(set(ANOS_INTERRUPCOES) - anos_ok)
if anos_faltando:
    print("ATENÇÃO: anos sem dados de interrupção nesta execução:", anos_faltando)
integ["tem_dados_interrupcao"] = ((integ.index >= inicio_int) & (integ.index <= cobertura_int[1])
                                  & integ.index.year.isin(list(anos_ok))
                                  & ~integ.index.to_period("M").isin(meses_lacuna)).astype(int) if len(diario_int) else 0
cols_int = list(diario_int.columns)
cobertos = integ["tem_dados_interrupcao"] == 1
integ.loc[cobertos, cols_int] = integ.loc[cobertos, cols_int].fillna(0)

# temperatura
integ = integ.join(temperatura, how="left")

# mensais com e sem defasagem
integ["mes_ref"] = integ.index.to_period("M").to_timestamp()
integ["mes_ref_lag"] = (integ.index.to_period("M") - DEFASAGEM_MENSAL).to_timestamp()
mensais = pd.concat([epe_pr, epe_sul, decfec_mensal], axis=1)
cols_preditoras = [c for c in mensais.columns if c.startswith("part_") or c in ("consumo_pr_total_mwh", "consumidores_pr",
                                                                               "dec_medio_conjuntos", "fec_medio_conjuntos")]
lag = mensais[cols_preditoras].add_suffix(f"_lag{DEFASAGEM_MENSAL}m")
integ = integ.join(lag, on="mes_ref_lag")
integ = integ.join(mensais, on="mes_ref")
transformar("Integração", "Bases mensais (EPE, DEC/FEC)", "Publicadas com atraso, usar o mês corrente vazaria informação",
            f"Junção pelo mês e versão preditora defasada em {DEFASAGEM_MENSAL} meses (sufixo _lag{DEFASAGEM_MENSAL}m)",
            "Só entra no modelo o que estaria disponível no dia da previsão", len(lag.columns))
integ = integ.drop(columns=["mes_ref_lag"])

# grande interrupção pela fração do PR atingida
if len(diario_int):
    limite_chi = integ.loc[cobertos, "chi_total"].quantile(PERCENTIL_GRANDE_INTERRUPCAO)
    integ["flag_chi_p99"] = ((integ["chi_total"] >= limite_chi) & cobertos).astype(int)
    if "consumidores_pr" in integ and integ["consumidores_pr"].notna().any():
        base_cons = integ["consumidores_pr"].ffill().bfill()
        integ["fracao_afetados"] = integ["consumidores_afetados"] / base_cons
        integ["flag_grande_interrupcao"] = ((integ["fracao_afetados"] >= LIMIAR_FRACAO_AFETADOS) & cobertos).astype(int)
        criterio = (f"afetações no dia >= {LIMIAR_FRACAO_AFETADOS:.0%} dos consumidores do {UF_DISTRIBUIDORA} (EPE), "
                    "e um consumidor atingido duas vezes conta duas vezes")
    else:
        integ["flag_grande_interrupcao"] = integ["flag_chi_p99"]
        criterio = f"CHI no percentil {PERCENTIL_GRANDE_INTERRUPCAO:.0%} ou acima (EPE indisponível)"
    transformar("Integração", "Interrupções diárias + consumidores do PR", "Identificar dias com demanda reprimida",
                f"flag_grande_interrupcao marca {criterio}. flag_chi_p99 (critério antigo) fica para comparação",
                "Nesses dias a demanda real foi maior que a carga registrada. O critério pela fração de consumidores "
                "separa melhor os dias em que a carga caiu (seção 9) e é mais fácil de explicar",
                int(integ["flag_grande_interrupcao"].sum()))
    print(f"Grande interrupção: {int(integ['flag_grande_interrupcao'].sum())} dias ({criterio})")
integ.index.name = "data"
print(f"Dataset integrado: {integ.shape[0]:,} dias x {integ.shape[1]} colunas")

### 8.1 Cobertura de cada coluna nova

In [ ]:
novas = [c for c in integ.columns if c not in ons.columns and c != "mes_ref"]
cob = []
for c in novas:
    s = integ[c]
    cob.append({"coluna": c, "primeiro_dia": s.first_valid_index(), "ultimo_dia": s.last_valid_index(),
                "pct_preenchido": round(100 * s.notna().mean(), 1)})
cobertura = pd.DataFrame(cob)
for _, r in cobertura.iterrows():
    qualidade.append({"fonte": "Integração", "dimensao": "Completude", "verificacao": f"Cobertura de {r['coluna']}",
                      "ocorrencias": None, "total_avaliado": len(integ), "percentual": r["pct_preenchido"],
                      "observacao": f"{r['primeiro_dia']:%d/%m/%Y} a {r['ultimo_dia']:%d/%m/%Y}" if pd.notna(r["primeiro_dia"]) else "sem dados"})
cobertura

## 9. Validação cruzada entre fontes

Fontes independentes que medem coisas relacionadas devem concordar. Cada verificação abaixo é uma evidência de qualidade da integração.

In [ ]:
validacoes = {}
spearman = lambda a, b: a.rank().corr(b.rank())   # Spearman pelos postos
pct = lambda x: "sem dias no critério" if pd.isna(x) else f"{br(x)}%"
br = lambda x, d=2: f"{x:,.{d}f}".replace(",", "X").replace(".", ",").replace("X", ".")

# (a) consumo da EPE x carga do ONS no Sul
ons_mensal = (integ["carga_mwmed"] * 24).groupby(integ["mes_ref"]).sum(min_count=1).rename("carga_ons_mwh")
dias_mes = integ.groupby("mes_ref").size()
completo = dias_mes == pd.Series(dias_mes.index.days_in_month, index=dias_mes.index)
comp = pd.concat([ons_mensal[completo], epe_sul], axis=1, join="inner").dropna() if len(epe_sul) else pd.DataFrame()
if len(comp) > 12:
    comp["razao"] = comp["consumo_sul_epe_mwh"] / comp["carga_ons_mwh"]
    r_var = comp[["carga_ons_mwh", "consumo_sul_epe_mwh"]].pct_change(12).corr().iloc[0, 1]
    validacoes["epe_ons_razao_media"] = comp["razao"].mean()
    validacoes["epe_ons_corr_var12m"] = r_var
    qualidade.append({"fonte": "Validação", "dimensao": "Acurácia", "verificacao": "Consumo EPE / carga ONS (Sul, mensal)",
                      "ocorrencias": None, "total_avaliado": len(comp), "percentual": round(100 * comp["razao"].mean(), 1),
                      "observacao": f"razão média {br(comp['razao'].mean(), 3)} em {len(comp)} meses e "
                                    f"correlação das variações em 12 meses de {br(r_var)}"})
    print(f"EPE/ONS: razão média {comp['razao'].mean():.3f}, correlação das variações em 12 meses {r_var:.2f}")

# (b) DEC oficial x CHI calculado
if len(decfec_mensal) and len(diario_int):
    mes_int = diario_int.index.to_period("M").to_timestamp()
    v = pd.concat([diario_int["chi_total"].groupby(mes_int).sum().rename("chi"),
                   diario_int["consumidores_afetados"].groupby(mes_int).sum().rename("afetados"),
                   decfec_mensal["dec_medio_conjuntos"]], axis=1, join="inner").dropna()
    if len(v) > 6:
        rho = spearman(v["chi"], v["dec_medio_conjuntos"])
        rho_af = spearman(v["afetados"], v["dec_medio_conjuntos"])
        validacoes["dec_chi_spearman"], validacoes["dec_afetados_spearman"] = rho, rho_af
        qualidade.append({"fonte": "Validação", "dimensao": "Consistência", "verificacao": "DEC oficial x extrato de interrupções (mensal, Spearman)",
                          "ocorrencias": None, "total_avaliado": len(v), "percentual": round(rho, 3),
                          "observacao": f"{br(rho)} com o CHI e {br(rho_af)} com os consumidores afetados, em {len(v)} meses. "
                                        "A concordância parcial é esperada porque o DEC oficial exclui os expurgos (grandes temporais) "
                                        "e aqui é média simples entre conjuntos, sem peso de consumidores"})
        print(f"DEC oficial x CHI: Spearman {rho:.2f}; x consumidores afetados: {rho_af:.2f} ({len(v)} meses)")

# (c) carga em dias de grande interrupção
if "flag_grande_interrupcao" in integ:
    # mediana das semanas vizinhas, sem o próprio dia
    s = integ["carga_mwmed"]
    esperado = pd.Series(np.nan, index=s.index)
    for dow in range(7):
        x = s[s.index.dayofweek == dow]
        esperado.loc[x.index] = pd.concat([x.shift(k) for k in (1, 2, 3, -1, -2, -3)], axis=1).median(axis=1)
    integ["desvio_carga_pct"] = 100 * (s / esperado - 1)
    # o mesmo para a temperatura
    if "temp_media_sul" in integ:
        t = integ["temp_media_sul"]; anom = pd.Series(np.nan, index=t.index)
        for dow in range(7):
            x = t[t.index.dayofweek == dow]
            anom.loc[x.index] = x - pd.concat([x.shift(k) for k in (1, 2, 3, -1, -2, -3)], axis=1).median(axis=1)
        integ["anomalia_temp_c"] = anom
    base_cob = integ[(integ["tem_dados_interrupcao"] == 1) & (integ["feriado"] == 0)].dropna(subset=["desvio_carga_pct"])
    d_grande = base_cob.loc[base_cob["flag_grande_interrupcao"] == 1, "desvio_carga_pct"].median()
    d_normal = base_cob.loc[base_cob["flag_grande_interrupcao"] == 0, "desvio_carga_pct"].median()
    top = base_cob.nlargest(10, "chi_total")
    rho_d = spearman(base_cob["chi_total"], base_cob["desvio_carga_pct"])
    validacoes.update({"desvio_grande": d_grande, "desvio_normal": d_normal, "desvio_top10": top["desvio_carga_pct"].median()})
    qualidade.append({"fonte": "Validação", "dimensao": "Acurácia", "verificacao": "Carga registrada em dias de grande interrupção (sem feriados)",
                      "ocorrencias": int(base_cob["flag_grande_interrupcao"].sum()), "total_avaliado": len(base_cob), "percentual": None,
                      "observacao": f"desvio mediano {pct(d_grande)} nos dias de grande interrupção contra {pct(d_normal)} nos demais, "
                                    f"{br(top['desvio_carga_pct'].median())}% nos 10 dias de maior CHI e Spearman CHI x desvio de {br(rho_d)}"})
    eventos = ", ".join(f"{d:%d/%m/%Y} ({br(r['desvio_carga_pct'], 1)}%)" for d, r in top.head(5).iterrows())
    qualidade.append({"fonte": "Validação", "dimensao": "Acurácia", "verificacao": "Os 5 dias de maior CHI e o desvio da carga",
                      "ocorrencias": None, "total_avaliado": None, "percentual": None, "observacao": eventos})
    print(f"Desvio mediano: {d_grande:+.2f}% (grandes) x {d_normal:+.2f}% (demais); 10 maiores: {top['desvio_carga_pct'].median():+.2f}%")
    print("Maiores eventos:", eventos)

    # efeito descontando a temperatura
    if "anomalia_temp_c" in integ:
        bt = base_cob.dropna(subset=["anomalia_temp_c"]).copy()
        coef = np.polyfit(bt["anomalia_temp_c"], bt["desvio_carga_pct"], 1)
        bt["desvio_sem_temp"] = bt["desvio_carga_pct"] - np.polyval(coef, bt["anomalia_temp_c"])
        integ["desvio_carga_sem_temp_pct"] = np.nan
        integ.loc[bt.index, "desvio_carga_sem_temp_pct"] = bt["desvio_sem_temp"]
        X = np.column_stack([np.ones(len(bt)), bt["anomalia_temp_c"], np.log10(bt["chi_total"].clip(lower=1))])
        beta, *_ = np.linalg.lstsq(X, bt["desvio_carga_pct"].values, rcond=None)
        res = bt["desvio_carga_pct"].values - X @ beta
        ep = np.sqrt(np.diag((res @ res / (len(bt) - 3)) * np.linalg.inv(X.T @ X)))
        q = bt["chi_total"].rank(pct=True)
        faixas = [("50% dos dias com menos interrupções", 0, .5), ("de 50% a 90%", .5, .9), ("de 90% a 99%", .9, .99),
                  ("1% com mais interrupções", .99, 1.01), ("0,1% com mais interrupções", .999, 1.01)]
        linhas_ef = [{"grupo": nome, "dias": int(((q >= lo) & (q < hi)).sum()),
                      "desvio_mediano_pct": bt.loc[(q >= lo) & (q < hi), "desvio_carga_pct"].median(),
                      "desvio_mediano_sem_temp_pct": bt.loc[(q >= lo) & (q < hi), "desvio_sem_temp"].median()}
                     for nome, lo, hi in faixas]
        if "fracao_afetados" in bt:
            for lim in (0.20, 0.30, 0.40):
                m = bt["fracao_afetados"] >= lim
                linhas_ef.append({"grupo": f"{lim:.0%} ou mais dos consumidores do PR afetados", "dias": int(m.sum()),
                                  "desvio_mediano_pct": bt.loc[m, "desvio_carga_pct"].median(),
                                  "desvio_mediano_sem_temp_pct": bt.loc[m, "desvio_sem_temp"].median()})
        efeito = pd.DataFrame(linhas_ef).round(2)
        efeito.to_csv(OUT / "efeito_interrupcoes_na_carga.csv", index=False)
        g_flag = bt["flag_grande_interrupcao"] == 1
        validacoes["efeito_temp_coef"] = coef[0]
        qualidade.append({"fonte": "Validação", "dimensao": "Acurácia",
                          "verificacao": "Carga em dias de grande interrupção, descontando a temperatura",
                          "ocorrencias": int(g_flag.sum()), "total_avaliado": len(bt), "percentual": None,
                          "observacao": f"{pct(bt.loc[g_flag, 'desvio_sem_temp'].median())} nos dias marcados contra "
                                        f"{pct(bt.loc[~g_flag, 'desvio_sem_temp'].median())} nos demais "
                                        f"(sem descontar, {pct(bt.loc[g_flag, 'desvio_carga_pct'].median())} contra "
                                        f"{pct(bt.loc[~g_flag, 'desvio_carga_pct'].median())})"})
        qualidade.append({"fonte": "Validação", "dimensao": "Acurácia",
                          "verificacao": "Regressão do desvio da carga = a + b x anomalia de temperatura + c x log10(CHI)",
                          "ocorrencias": None, "total_avaliado": len(bt), "percentual": None,
                          "observacao": f"b = {br(beta[1])} (erro padrão {br(ep[1])}) pontos percentuais por °C e "
                                        f"c = {br(beta[2])} (erro padrão {br(ep[2])})"})
        print(f"Cada 1 °C acima dos dias vizinhos: {beta[1]:+.2f} p.p. na carga")
        print(efeito.to_string(index=False))

# (d) carga x temperatura por estação
if "temp_media_sul" in integ:
    uteis = integ[(integ["fim_de_semana"] == 0) & (integ["feriado"] == 0)]
    corr_est = pd.Series({e: g["carga_mwmed"].corr(g["temp_media_sul"]) for e, g in uteis.groupby("estacao")})
    validacoes["corr_temp_estacao"] = corr_est.round(2).to_dict()
    print("Correlação carga x temperatura (dias úteis), por estação:", corr_est.round(2).to_dict())

## 10. Figuras

In [ ]:
# figura 1, carga x temperatura
if "temp_media_sul" in integ:
    u = integ[(integ["fim_de_semana"] == 0) & (integ["feriado"] == 0)].dropna(subset=["temp_media_sul", "carga_mwmed"])
    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.scatter(u["temp_media_sul"], u["carga_mwmed"], s=6, alpha=0.35, color=COR_SERIE, label="Dia útil")
    faixas = pd.cut(u["temp_media_sul"], bins=np.arange(np.floor(u["temp_media_sul"].min()), u["temp_media_sul"].max() + 1, 1))
    m = u.groupby(faixas, observed=True)["carga_mwmed"].median()
    ax.plot([i.mid for i in m.index], m.values, color=COR_2, linewidth=2.5, label="Mediana por faixa de 1 °C")
    ax.set_xlabel("Temperatura média das capitais do Sul (°C)"); ax.set_ylabel("Carga (MWmed)")
    ax.set_title("Carga diária x temperatura - dias úteis, subsistema Sul")
    ax.legend(frameon=False, loc="upper left")
    fig.tight_layout(); fig.savefig(FIG / "fig1_carga_x_temperatura.png"); plt.show()

# figura 2, CHI diário
if len(diario_int):
    c = integ.loc[integ["tem_dados_interrupcao"] == 1]
    fig, ax = plt.subplots()
    ax.plot(c.index, c["chi_total"] / 1e3, color=COR_SERIE, linewidth=0.6, label="CHI diário")
    g = c[c["flag_grande_interrupcao"] == 1]
    ax.scatter(g.index, g["chi_total"] / 1e3, s=16, color=COR_ALERTA, zorder=3,
               label=f"Grande interrupção ({LIMIAR_FRACAO_AFETADOS:.0%} ou mais dos consumidores afetados)")
    ax.set_yscale("log"); ax.set_ylabel("Mil consumidores-hora (escala log)")
    ax.set_title("Interrupções na Copel: consumidor-hora interrompido por dia", pad=28)
    ax.legend(frameon=False, loc="lower left", bbox_to_anchor=(0, 1.0), ncol=2)
    fig.tight_layout(); fig.savefig(FIG / "fig2_chi_diario_copel.png"); plt.show()

# figura 3, desvio da carga
if "desvio_carga_sem_temp_pct" in integ:
    b = integ[(integ["tem_dados_interrupcao"] == 1) & (integ["feriado"] == 0)]
    grupos = [b.loc[b["flag_grande_interrupcao"] == 0, "desvio_carga_sem_temp_pct"].dropna(),
              b.loc[b["flag_grande_interrupcao"] == 1, "desvio_carga_sem_temp_pct"].dropna()]
    fig, ax = plt.subplots(figsize=(7, 4))
    estilo = dict(showfliers=False, patch_artist=True, boxprops=dict(facecolor="#D6E4EF", edgecolor=COR_SERIE),
                  medianprops=dict(color=COR_SERIE, linewidth=1.5), whiskerprops=dict(color=COR_REF), capprops=dict(color=COR_REF))
    rotulos = ["Demais dias", "Dias de grande\ninterrupção"]
    try:
        ax.boxplot(grupos, tick_labels=rotulos, **estilo)
    except TypeError:
        ax.boxplot(grupos, labels=rotulos, **estilo)
    ax.axhline(0, color=COR_REF, linewidth=0.8)
    ax.set_ylabel("Desvio da carga, descontada a temperatura (%)")
    ax.set_title("Carga registrada em dias de grande interrupção")
    fig.tight_layout(); fig.savefig(FIG / "fig3_carga_dias_interrupcao.png"); plt.show()

# figura 4, classes de consumo no PR
cols_part = [c for c in ("part_residencial_pr", "part_industrial_pr", "part_comercial_pr") if c in epe_pr]
if cols_part:
    fig, ax = plt.subplots()
    cores = {"part_residencial_pr": COR_SERIE, "part_industrial_pr": COR_2, "part_comercial_pr": "#5B8C5A"}
    nomes = {"part_residencial_pr": "Residencial", "part_industrial_pr": "Industrial", "part_comercial_pr": "Comercial"}
    for c in cols_part:
        s = epe_pr.loc[epe_pr.index >= ini_ons, c] * 100
        ax.plot(s.index, s.values, color=cores[c], linewidth=1.6, label=nomes[c])
    ax.set_ylabel("Participação no consumo (%)"); ax.set_title(f"Consumo de energia por classe - {UF_DISTRIBUIDORA} (EPE)", pad=28)
    ax.legend(frameon=False, loc="lower left", bbox_to_anchor=(0, 1.0), ncol=3)
    fig.tight_layout(); fig.savefig(FIG / "fig4_participacao_classes_pr.png"); plt.show()

# figura 5, EPE x ONS
if len(comp):
    fig, ax = plt.subplots()
    ax.plot(comp.index, comp["carga_ons_mwh"] / 1e6, color=COR_SERIE, linewidth=1.6, label="Carga ONS (energia entregue)")
    ax.plot(comp.index, comp["consumo_sul_epe_mwh"] / 1e6, color=COR_2, linewidth=1.6, label="Consumo faturado EPE")
    ax.set_ylabel("TWh por mês"); ax.set_title("Região Sul: carga do ONS x consumo da EPE")
    ax.legend(frameon=False, loc="upper left")
    fig.tight_layout(); fig.savefig(FIG / "fig5_epe_x_ons_sul.png"); plt.show()

## 11. Saídas com dataset integrado, dicionário com linhagem, relatórios e resumo

In [ ]:
LINHAGEM = {
    "chi_total": ("ANEEL interrupções", "Consumidor-hora interrompido no dia (Copel)", "Sinalização do alvo, não preditor"),
    "chi_nao_prog": ("ANEEL interrupções", "CHI de interrupções não programadas", "Sinalização do alvo, não preditor"),
    "n_interrupcoes": ("ANEEL interrupções", "Número de interrupções iniciadas no dia", "Sinalização do alvo, não preditor"),
    "n_interrupcoes_nao_prog": ("ANEEL interrupções", "Interrupções não programadas no dia", "Sinalização do alvo, não preditor"),
    "consumidores_afetados": ("ANEEL interrupções", "Soma de consumidores afetados no dia", "Sinalização do alvo, não preditor"),
    "tem_dados_interrupcao": ("ANEEL interrupções", "1 se o dia está no período coberto pela base de interrupções", "Controle de cobertura"),
    "flag_grande_interrupcao": ("ANEEL interrupções + EPE", "1 se as afetações do dia somam 20% ou mais dos consumidores do PR", "Exclusão ou ponderação no treino"),
    "flag_chi_p99": ("ANEEL interrupções", "1 se o CHI do dia está no percentil 99 ou acima (critério antigo)", "Comparação, não usar no modelo"),
    "fracao_afetados": ("ANEEL interrupções + EPE", "Consumidores afetados no dia / consumidores do PR (afetações repetidas contam)", "Sinalização do alvo, não preditor"),
    "anomalia_temp_c": ("Calculado (Open-Meteo)", "Temperatura do dia menos a mediana do mesmo dia da semana nas semanas vizinhas (°C)", "Análise, não preditor"),
    "desvio_carga_sem_temp_pct": ("Calculado", "Desvio da carga descontado o efeito da anomalia de temperatura (%)", "Análise, não preditor"),
    "temp_media_sul": ("Open-Meteo (ERA5)", "Temperatura média das 3 capitais (°C)", "Preditor (em produção vem da previsão do tempo)"),
    "temp_max_sul": ("Open-Meteo (ERA5)", "Temperatura máxima média das 3 capitais (°C)", "Preditor (em produção vem da previsão do tempo)"),
    "temp_min_sul": ("Open-Meteo (ERA5)", "Temperatura mínima média das 3 capitais (°C)", "Preditor (em produção vem da previsão do tempo)"),
    "temp_media_curitiba": ("Open-Meteo (ERA5)", "Temperatura média de Curitiba (°C)", "Preditor (em produção vem da previsão do tempo)"),
    "graus_dia_calor": ("Open-Meteo (ERA5)", "Graus acima de 24 °C na média do dia", "Preditor (em produção vem da previsão do tempo)"),
    "graus_dia_frio": ("Open-Meteo (ERA5)", "Graus abaixo de 18 °C na média do dia", "Preditor (em produção vem da previsão do tempo)"),
    "desvio_carga_pct": ("Calculado", "Desvio da carga em relação ao mesmo dia da semana nas semanas vizinhas (%)", "Análise, não preditor"),
    "mes_ref": ("Calculado", "Primeiro dia do mês, chave para bases mensais", "Chave de junção"),
}
for c in integ.columns:
    if c in LINHAGEM:
        continue
    if c.endswith(f"_lag{DEFASAGEM_MENSAL}m"):
        fonte = "ANEEL DEC/FEC" if c.startswith(("dec_", "fec_")) else "EPE"
        LINHAGEM[c] = (fonte, f"{c.rsplit('_lag', 1)[0]} de {DEFASAGEM_MENSAL} meses antes", "Preditor (respeita a publicação)")
    elif c.startswith(("consumo_pr", "part_", "consumidores_pr")):
        LINHAGEM[c] = ("EPE", f"{c} no mês corrente", "Análise, não preditor (vazaria informação)")
    elif c == "consumo_sul_epe_mwh":
        LINHAGEM[c] = ("EPE", "Consumo faturado do Sul no mês (MWh)", "Validação cruzada")
    elif c.startswith(("dec_", "fec_", "n_conjuntos")):
        LINHAGEM[c] = ("ANEEL DEC/FEC", f"{c} no mês corrente", "Validação e acompanhamento, não preditor")

dataset = integ.reset_index()
dataset["data"] = dataset["data"].dt.strftime("%Y-%m-%d")
dataset["mes_ref"] = pd.to_datetime(dataset["mes_ref"]).dt.strftime("%Y-%m")
dataset.to_csv(OUT / "dataset_integrado.csv", index=False)

dic = []
for c in dataset.columns:
    fonte, sig, uso = LINHAGEM.get(c, ("ONS (Etapa 2)", "ver dicionário da Etapa 2", "conforme Etapa 2"))
    s = integ[c] if c in integ.columns else dataset[c]
    dic.append({"campo": c, "fonte": fonte, "significado": sig, "uso_no_modelo": uso,
                "pct_preenchido": round(100 * s.notna().mean(), 1)})
dicionario = pd.DataFrame(dic)
dicionario.to_csv(OUT / "dicionario_integrado.csv", index=False)

pd.DataFrame(qualidade).to_csv(OUT / "relatorio_qualidade_fontes.csv", index=False)
pd.DataFrame(log).to_csv(OUT / "log_transformacoes_etapa3.csv", index=False)
pd.DataFrame(registro_downloads).to_csv(OUT / "registro_downloads_etapa3.csv", index=False)
df_anos.to_csv(OUT / "interrupcoes_por_ano.csv", index=False)
cobertura.to_csv(OUT / "cobertura_colunas.csv", index=False)
print(f"Dataset integrado: {dataset.shape[0]:,} linhas x {dataset.shape[1]} colunas")
dicionario

In [ ]:
br = lambda x, d=2: f"{x:,.{d}f}".replace(",", "X").replace(".", ",").replace("X", ".")
inteiro = lambda n: f"{int(n):,}".replace(",", ".")
L = [f"# Resumo da integração (Etapa 3)\n\nExecução em {DATA_EXECUCAO:%d/%m/%Y %H:%M} (horário UTC do Colab)\n",
     "## Fontes"]
L.append(f"- ONS (Etapa 2) com {inteiro(len(ons))} dias, de {ini_ons:%d/%m/%Y} a {fim_ons:%d/%m/%Y}.")
if len(diario_int):
    L.append(f"- ANEEL interrupções da Copel com {inteiro(len(iv))} interrupções em {inteiro(len(diario_int))} dias, "
             f"de {cobertura_int[0]:%d/%m/%Y} a {cobertura_int[1]:%d/%m/%Y}.")
if len(decfec_mensal):
    L.append(f"- ANEEL DEC/FEC da Copel com {len(decfec_mensal)} meses, de {decfec_mensal.index.min():%m/%Y} a {decfec_mensal.index.max():%m/%Y}.")
if len(epe_pr):
    L.append(f"- EPE consumo por classe no {UF_DISTRIBUIDORA}, de {epe_pr.index.min():%m/%Y} a {epe_pr.index.max():%m/%Y}.")
if len(temperatura):
    L.append(f"- Open-Meteo com {inteiro(temperatura['temp_media_sul'].notna().sum())} dias de temperatura.")
L.append("\n## Qualidade e validação")
for _, r in pd.DataFrame(qualidade).iterrows():
    if r["fonte"] == "Integração":
        continue
    if r["fonte"] != "Validação" and pd.notna(r["total_avaliado"]) and pd.notna(r["ocorrencias"]):
        L.append(f"- {r['fonte']}, {r['verificacao']}, {inteiro(r['ocorrencias'])} de {inteiro(r['total_avaliado'])} "
                 f"({br(r['percentual'])}%). {r['observacao']}".rstrip())
    else:
        L.append(f"- {r['fonte']}, {r['verificacao']}, {r['observacao']}")
L.append("\n## Cobertura das colunas novas")
for _, r in cobertura.iterrows():
    per = f"{r['primeiro_dia']:%d/%m/%Y} a {r['ultimo_dia']:%d/%m/%Y}" if pd.notna(r["primeiro_dia"]) else "sem dados"
    L.append(f"- {r['coluna']} com {br(r['pct_preenchido'], 1)}% preenchido ({per})")
if "corr_temp_estacao" in validacoes:
    L.append("\n## Carga x temperatura (dias úteis)")
    L.append("- Correlação por estação, " + ", ".join(f"{k} {br(v)}" for k, v in validacoes["corr_temp_estacao"].items()))
L.append(f"\n## Dataset integrado\n- {inteiro(dataset.shape[0])} dias x {dataset.shape[1]} colunas e {len(log)} transformações registradas.")
resumo = "\n".join(L)
(OUT / "resumo_etapa3.md").write_text(resumo, encoding="utf-8")
(OUT / "parametros_execucao_etapa3.json").write_text(json.dumps({
    "ANOS_INTERRUPCOES": ANOS_INTERRUPCOES, "PADRAO_AGENTE": PADRAO_AGENTE, "UF_DISTRIBUIDORA": UF_DISTRIBUIDORA,
    "DEFASAGEM_MENSAL": DEFASAGEM_MENSAL, "LIMIAR_FRACAO_AFETADOS": LIMIAR_FRACAO_AFETADOS,
    "PERCENTIL_GRANDE_INTERRUPCAO": PERCENTIL_GRANDE_INTERRUPCAO,
    "DURACAO_MAX_PLAUSIVEL_H": DURACAO_MAX_PLAUSIVEL_H, "CIDADES": CIDADES,
    "data_execucao": DATA_EXECUCAO.isoformat(timespec="seconds"),
    "python": platform.python_version(), "pandas": pd.__version__, "numpy": np.__version__}, indent=2, ensure_ascii=False), encoding="utf-8")
print(resumo)

## 12. Baixar os resultados no Colab

In [ ]:
pasta = BASE / "_entrega"
if pasta.exists():
    shutil.rmtree(pasta)
pasta.mkdir()
shutil.copytree(OUT, pasta / "saidas")
shutil.copytree(FIG, pasta / "figuras")
arquivo_zip = shutil.make_archive("etapa3_resultados", "zip", pasta)
shutil.rmtree(pasta)
print("Gerado:", arquivo_zip)
try:
    from google.colab import files
    files.download(arquivo_zip)
except ImportError:
    pass